In [0]:
# Workshop configuration — sourced via %run ../utils/config from all participant notebooks
#
# Sets the following variables in the calling notebook's scope:
#   catalog         - Unity Catalog catalog name (shared, fixed)
#   shared_schema   - Schema containing pre-staged shared data
#   schema          - Personal schema for this participant
#   safe_username   - Sanitized username (lowercase, non-alphanumeric → _)
#   volume_path     - Personal Volume path for uploading documents
#   shared_volume   - Shared Volume path containing all pre-staged PDFs

import re

shared_schema = "00_shared"
try:
    catalog
except NameError:
    catalog = "vectorcatalog01" 

spark.sql(f"USE CATALOG {catalog}")

# Get the current user's email from Spark
username = spark.sql("SELECT current_user()").first()[0]

# Strip off anything after '@'
safe_username = username.split('@')[0]

# Sanitize for use as a schema name (replace non-alphanumeric chars with _)
safe_username = re.sub(r'[^a-z0-9]', '_', safe_username.lower()).strip('_')

try:
    schema
except NameError:
    schema = safe_username

# Schema won't exist yet during participant setup
try:
    spark.sql(f"USE SCHEMA {schema}")
except:
    print(f"Schema {schema} doesn't exist yet.  To be expected when running 00_setup_participant notebook")

shared_volume = f"/Volumes/{catalog}/{shared_schema}/financial_docs_raw"
volume_path = f"/Volumes/{catalog}/{schema}/financial_docs"

print("="*50)
print(f"Your Databricks login (username)      : {username}")
print(f"Your safe username (safe_username)    : {safe_username}")
print(f"Your schema name (catalog).(schema)   : {catalog}.{schema}")
print(f"Your volume (volume_path)             : {volume_path}")
print("\n")
print(f"Shared schema name (catalog).(shared_schema) : {catalog}.{shared_schema}")
print(f"Shared volume path (shared_volume)           : {shared_volume}")
print('=' * 50)
print("\n")

In [0]:
def setup_participant_env():
    """Create personal schema and volume in the workshop catalog.
    """

    catalog_quoted = f"`{catalog}`"

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_quoted}.{schema}")
    print(f"✓ Schema ready:  {catalog}.{schema}")

    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.financial_docs"
    )
    print(f"✓ Volume ready:  {volume_path}")

